In [ ]:
import sys, warnings
sys.path.append("..")
warnings.filterwarnings("ignore")   # hides harmless convergence warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from src.utils import evaluate_forecasts

df = pd.read_csv("../data/GSPC_processed.csv", index_col=0, parse_dates=True)
r = df["log_return"]
split = int(len(r) * 0.8)            # same split as notebooks 03 and 04

specs = {
    "GARCH(1,1)": dict(vol="GARCH", p=1, q=1),
    "EGARCH":     dict(vol="EGARCH", p=1, o=1, q=1),
    "GJR-GARCH":  dict(vol="GARCH", p=1, o=1, q=1),
}
REFIT_EVERY = 20                      # trading days between re-estimations

In [ ]:
n = len(r)
parts = {name: [] for name in specs}

for i, s in enumerate(range(split, n, REFIT_EVERY)):
    e = min(s + REFIT_EVERY, n)
    for name, spec in specs.items():
        am = arch_model(r.iloc[:e], mean="Constant", dist="t", **spec)
        res = am.fit(last_obs=r.index[s - 1], disp="off")
        f = res.forecast(horizon=1, start=r.index[s - 1], reindex=False).variance["h.1"]
        parts[name].append(f.shift(1).loc[r.index[s:e]])
    if i % 10 == 0:
        print(f"refit block {i}, up to {r.index[s].date()}")

rolling = pd.DataFrame({f"{name} (rolling)": pd.concat(v) for name, v in parts.items()})
rolling.head()

In [ ]:
fixed = pd.read_csv("../data/garch_forecasts.csv", index_col=0, parse_dates=True)
fixed = fixed.rename(columns=lambda c: c if c.startswith("Rolling 21d") else f"{c} (fixed)")

all_fc = fixed.join(rolling, how="inner").dropna()
proxy = r.loc[all_fc.index] ** 2
evaluate_forecasts(proxy, all_fc).round(4)

In [ ]:
from pathlib import Path
if Path("../data/lstm_forecasts.csv").exists():
    lstm = pd.read_csv("../data/lstm_forecasts.csv", index_col=0, parse_dates=True)["LSTM"]
    all_fc = all_fc.join(lstm, how="inner").dropna()
    proxy = r.loc[all_fc.index] ** 2
    print("LSTM included")
evaluate_forecasts(proxy, all_fc).round(4)

In [ ]:
import statsmodels.api as sm

def dm_test(proxy, fa, fb, lags=5):
    fa, fb = fa.clip(lower=1e-8), fb.clip(lower=1e-8)
    d = ((np.log(fa) + proxy / fa) - (np.log(fb) + proxy / fb)).values
    res = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return res.tvalues[0], res.pvalues[0]

names = list(all_fc.columns)
stat = pd.DataFrame(index=names, columns=names, dtype=float)
pval = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        if a != b:
            stat.loc[a, b], pval.loc[a, b] = dm_test(proxy, all_fc[a], all_fc[b])

print("DM statistic (negative = ROW model has lower loss than COLUMN model)")
display(stat.round(2))
print("p-values (below 0.05 = difference is statistically significant)")
display(pval.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(stat.values.astype(float), cmap="RdBu", vmin=-4, vmax=4)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        if i != j:
            star = "*" if pval.iloc[i, j] < 0.05 else ""
            ax.text(j, i, f"{stat.iloc[i, j]:.1f}{star}", ha="center", va="center", fontsize=8)
ax.set_title("Diebold-Mariano statistic (row vs column). Blue = row better, * p<0.05")
plt.colorbar(im); plt.tight_layout(); plt.show()